In [ ]:
import os
import requests

# 1. Definisikan IP masing-masing gateway
gateways = {
    "main": "192.168.18.14",
    "riset": "192.168.18.22"
}

# Pastikan folder logs/ lokal ada
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

# 2. Proses sinkronisasi otomatis via WiFi
for name, ip in gateways.items():
    print(f"\n===== MENGHUBUNGKAN KE GATEWAY {name.upper()} ({ip}) =====")
    try:
        # A. Ambil daftar berkas CSV di MicroSD melalui endpoint /list
        list_url = f"http://{ip}/list?dir=datalog"
        response = requests.get(list_url, timeout=5)
        
        if response.status_code == 200:
            # Pecah response teks menjadi daftar nama file
            filenames = [f.strip() for f in response.text.split('\n') if f.strip().endswith('.csv')]
            print(f"Daftar file di MicroSD: {filenames}")
            
            # B. Unduh setiap berkas melalui endpoint /download
            for filename in filenames:
                # Beri akhiran nama gateway agar tidak saling menimpa
                local_name = f"{filename.replace('.csv', '')}_{name}.csv"
                local_path = os.path.join(log_dir, local_name)
                
                print(f"Mengunduh {filename} -> {local_name}...", end="")
                download_url = f"http://{ip}/download?file=/datalog/{filename}"
                file_response = requests.get(download_url, timeout=10)
                
                if file_response.status_code == 200:
                    with open(local_path, "wb") as f:
                        f.write(file_response.content)
                    print(" [Sukses]")
                else:
                    print(f" [Gagal (HTTP {file_response.status_code})]")
        else:
            print(f"Gagal memindai folder (HTTP {response.status_code})")
            
    except Exception as e:
        print(f"Koneksi gagal ke {ip} (Gateway offline atau berbeda WiFi): {e}")

print("\n===== SINKRONISASI SELESAI =====")


In [ ]:
import os
import pandas as pd

log_dir = "datalogs"
if not os.path.exists(log_dir):
    print(f"Directory '{log_dir}' not found.")
else:
    csv_files = [f for f in os.listdir(log_dir) if f.endswith('.csv')]
    
    all_data = []
    for file in csv_files:
        file_path = os.path.join(log_dir, file)
        try:
            # Handle potentially malformed CSVs
            df = pd.read_csv(file_path, on_bad_lines='skip')
            
            # Clean column names (strip spaces)
            df.columns = df.columns.str.strip()
            
            # Convert unix timestamp to readable datetime
            if 'Timestamp' in df.columns:
                # Convert to numeric first, coercing errors to NaN
                df['Timestamp'] = pd.to_numeric(df['Timestamp'], errors='coerce')
                # Convert unix timestamp (seconds) to readable datetime
                df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s', errors='coerce')
                
                # Adjust timezone to Asia/Jakarta (UTC+7)
                df['Datetime'] = df['Datetime'] + pd.Timedelta(hours=7)
                
            df['Source_File'] = file
            all_data.append(df)
            print(f"Successfully processed {file} - Rows: {len(df)}")
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Sort by datetime if it exists
        if 'Datetime' in combined_df.columns:
            combined_df = combined_df.sort_values(by='Datetime', ascending=False)
            
            # Reorder columns to put Datetime first
            cols = combined_df.columns.tolist()
            if 'Datetime' in cols:
                cols.insert(0, cols.pop(cols.index('Datetime')))
                combined_df = combined_df[cols]
            
        display(combined_df.head(50))
    else:
        print("No valid data found in CSV files.")


### Analisis & Visualisasi Data Hilang (*Missing Values*) pada File Log CSV
Sel berikut secara otomatis memindai seluruh file rekaman di dalam folder `logs/`, menggabungkannya menjadi satu matriks besar, dan memvisualisasikan tumpukan data yang hilang (baris bernilai Null/NaN) untuk memonitor kesehatan dan reliabilitas transmisi sensor IoT.

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Membaca semua file CSV di dalam folder logs
log_files = glob.glob('datalogs/*.csv')
dfs = []
for file in log_files:
    try:
        df = pd.read_csv(file, low_memory=False, on_bad_lines='skip', index_col=False)
        dfs.append(df)
        print(f"Berhasil memuat {os.path.basename(file)} dengan {len(df)} baris.")
    except Exception as e:
        print(f"Error memuat {file}: {e}")

if dfs:
    # Menggabungkan semua dataframe
    df_all = pd.concat(dfs, ignore_index=True)
    
    # Menghitung jumlah NaN/Null di setiap kolom
    missing_data = df_all.isnull().sum().sort_values(ascending=False)
    
    # Hanya ambil kolom yang memiliki missing value > 0
    missing_data_positive = missing_data[missing_data > 0]
    
    print("\nRincian Total Missing Values:")
    print(missing_data_positive if not missing_data_positive.empty else "Tidak ada missing value di seluruh file log!")
    
    # Visualisasi Bar Chart
    if not missing_data_positive.empty:
        plt.figure(figsize=(14, 6))
        sns.barplot(x=missing_data_positive.index, y=missing_data_positive.values, palette='Reds_r')
        plt.xticks(rotation=45, ha='right')
        plt.title('Visualisasi Data yang Hilang (Missing Values) pada Sensor Logs', fontsize=14, fontweight='bold')
        plt.ylabel('Jumlah Baris Kosong', fontsize=12)
        plt.xlabel('Nama Kolom Sensor', fontsize=12)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print("\nDataset log Anda lengkap dan bersih dari Null/NaN. Visualisasi tidak diperlukan.")
else:
    print("Tidak ditemukan file CSV di dalam folder logs.")

### Analisis Kekosongan Transmisi Temporal Per Node (Resampling & `missingno`)
Fungsi ini membaca file CSV mentah (`Timestamp` unix) lalu membedah sinyal transmisi berdasakan tipe **`NodeID` (Sensor Node 1 vs Node 3)** secara terpisah. 

Setiap node di-*resample* secara ketat ke interval **1 Menit (`1T`)** untuk memaksa waktu yang terputus (sensor mati/sinyal hilang) terekspos menjadi nilai kosong. Garis putih tebal di dalam matriks `missingno` menandakan di menit ke berapa sebuah Node kehilangan koneksinya dari sistem pusat.

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import missingno as msno
import os
import warnings
warnings.filterwarnings("ignore")

# Membaca semua file CSV di dalam folder logs
log_files = glob.glob('datalogs/*.csv')

for file in log_files:
    try:
        df = pd.read_csv(file, low_memory=False, on_bad_lines='skip', index_col=False)
        
        # Generate Datetime from Timestamp if missing
        if 'Datetime' not in df.columns and 'Timestamp' in df.columns:
            df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Jakarta').dt.tz_localize(None)
            
        if 'Datetime' in df.columns and 'NodeID' in df.columns:
            df['Datetime'] = pd.to_datetime(df['Datetime'])
            nodes = sorted(df['NodeID'].dropna().unique())
            
            for node in nodes:
                # Pisahkan data per NodeID
                df_node = df[df['NodeID'] == node].copy()
                df_node = df_node.drop_duplicates(subset=['Datetime'])
                df_node = df_node.set_index('Datetime')
                df_node = df_node.sort_index()
                
                # Resample per node (1 Menit interval)
                df_resampled = df_node.resample('1T').asfreq()
                
                print(f"\n{'='*70}")
                print(f"📁 File: {os.path.basename(file)}  |  📡 Sensor: Node {int(node)}")
                print(f"⏱ Rentang: {df_resampled.index.min()} hingga {df_resampled.index.max()}")
                print(f"📊 Ekspektasi Total: {len(df_resampled)} baris")
                print(f"❌ Missing Rows (Mati/Gagal Kirim): {df_resampled['Timestamp'].isnull().sum()}")
                print(f"{'='*70}")
                
                # Beda warna per node agar mudah dibedakan
                color = (0.2, 0.4, 0.6) if node == 1 else (0.7, 0.4, 0.2)
                
                msno.matrix(df_resampled, figsize=(14, 5), sparkline=False, fontsize=10, color=color)
                plt.title(f"Visualisasi Transmisi & Data Putus - Node {int(node)} ({os.path.basename(file)})", fontsize=16, fontweight='bold', pad=20)
                plt.tight_layout()
                plt.show()
                
        else:
            print(f"⚠️ Kolom 'Datetime'/'Timestamp' atau 'NodeID' tidak lengkap pada {os.path.basename(file)}")
    except Exception as e:
        print(f"Error memproses {file}: {e}")

### Konversi *Unix Epoch* ke Format Tanggal Manusia (*Readable Datetime*)
Fungsi otomatis di bawah ini didedikasikan untuk memindai setiap log mentah di dalam folder `logs/` dan menduplikasi/meng-*copy* file tersebut dengan nama awalan `readable_`.

Konversi ini akan menyingkirkan angka detik *Unix Timestamp* yang membingungkan dan merubahnya menjadi waktu **Asia/Jakarta** (misal: `2026-06-15 15:30:00`), lalu menyimpannya di kolom paling pertama, mempermudah inspeksi manual kita.

In [ ]:
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings("ignore")

# Membaca semua file CSV di dalam folder logs
log_files = glob.glob('datalogs/*.csv')

print(f"{'='*60}")
print("🔄 Memulai Proses Konversi Timestamp ke Readable Datetime")
print(f"{'='*60}\n")

for file in log_files:
    basename = os.path.basename(file)
    
    # Abaikan file yang sudah dikonversi sebelumnya
    if basename.startswith('readable_'):
        continue
        
    output_path = os.path.join(os.path.dirname(file), f"readable_{basename}")
    
    try:
        # Membaca data dengan mengabaikan baris cacat
        df = pd.read_csv(file, low_memory=False, on_bad_lines='skip', index_col=False)
        
        # Mengecek apakah file memiliki Timestamp
        if 'Timestamp' in df.columns:
            # Mengonversi waktu epoch (detik) ke zona waktu Asia/Jakarta
            df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Jakarta').dt.tz_localize(None)
            
            # Pindahkan kolom Datetime ke paling depan agar mudah dibaca manusia
            cols = df.columns.tolist()
            if 'Datetime' in cols:
                cols.insert(0, cols.pop(cols.index('Datetime')))
                df = df[cols]
            
            # Simpan file sebagai copy
            df.to_csv(output_path, index=False)
            print(f"✅ Berhasil membuat salinan: {output_path} ({len(df)} baris)")
        else:
            print(f"⚠️ Dilewati (Tidak ada 'Timestamp'): {basename}")
            
    except Exception as e:
        print(f"❌ Error memproses {basename}: {e}")

print(f"\n{'='*60}")
print("🎯 Konversi Selesai! Silakan cek folder logs/ Anda.")
print(f"{'='*60}")